In [1]:
import pandas as pd
import numpy as np
import chromadb
from tqdm import tqdm

In [2]:
import pyarrow.parquet as pq
import chromadb
from tqdm import tqdm

parquet_file = pq.ParquetFile("../data/raw/complaint_embeddings.parquet")
print("Total rows:", parquet_file.metadata.num_rows)

Total rows: 1375327


In [3]:
import shutil
import os

if os.path.exists("../vector_store"):
    shutil.rmtree("../vector_store")
print("Cleared old vector_store directory.")

Cleared old vector_store directory.


In [4]:
import pyarrow.parquet as pq
import chromadb
from tqdm import tqdm

parquet_file = pq.ParquetFile("../data/raw/complaint_embeddings.parquet")

client = chromadb.PersistentClient(path="../vector_store")
collection = client.create_collection(name="complaint_chunks")

MAX_ROWS = 50000
BATCH_SIZE = 1000
total_indexed = 0

for batch in tqdm(parquet_file.iter_batches(batch_size=BATCH_SIZE)):
    if total_indexed >= MAX_ROWS:
        break

    batch_df = batch.to_pandas()
    ids = batch_df['id'].astype(str).tolist()
    documents = batch_df['document'].astype(str).tolist()
    embeddings = [list(e) for e in batch_df['embedding']]

    metadatas = [
        {
            "complaint_id": str(m.get('complaint_id', '')),
            "product_category": str(m.get('product_category', '')),
            "product": str(m.get('product', '')),
            "issue": str(m.get('issue', '')),
            "sub_issue": str(m.get('sub_issue', '')),
            "company": str(m.get('company', '')),
            "state": str(m.get('state', '')),
            "date_received": str(m.get('date_received', '')),
            "chunk_index": int(m.get('chunk_index', 0)) if m.get('chunk_index') is not None else 0,
            "total_chunks": int(m.get('total_chunks', 1)) if m.get('total_chunks') is not None else 1,
        }
        for m in batch_df['metadata']
    ]

    collection.add(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)
    total_indexed += len(ids)

print(f"Indexed {total_indexed} chunks. Collection count: {collection.count()}")

50it [04:25,  5.31s/it]


Indexed 50000 chunks. Collection count: 50000


In [6]:
print(collection.count())

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

test_q = "Why are people unhappy with Credit Cards?"
query_embedding = embedder.encode([test_q]).tolist()
results = collection.query(query_embeddings=query_embedding, n_results=5)

for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(meta['product_category'], '|', meta['complaint_id'])
    print(doc[:150], '\n')

50000


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Credit Card | 12682005
credit card services harming the elderly and . 

Credit Card | 11527472
ments were due to a natural disaster in my inability to get to a bank. normally, i wouldve made a deposit on , but that is when the natural disaster i 

Credit Card | 13012072
well so this will help a lot as ive noted to them. this feels retaliatory, especially following my apr request, and harmful to customers like me who a 

Credit Card | 12032937
here actions when it comes to using a credit card. 

Credit Card | 10404794
justification of their enablement of credit card fraud. 



In [15]:
def retrieve(question, top_k=5, product_filter=None):
    query_embedding = embedder.encode([question]).tolist()
    where = {"product_category": product_filter} if product_filter else None
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=where
    )
    return {
        "documents": results["documents"][0],
        "metadatas": results["metadatas"][0],
    }

test = retrieve("Why are people unhappy with Credit Cards?")

In [10]:
PROMPT_TEMPLATE = """You are a financial analyst assistant for CrediTrust. Your task is to answer questions \
about customer complaints. Use the following retrieved complaint excerpts to formulate \
your answer. If the context doesn't contain the answer, state that you don't have \
enough information. Do not invent details that are not present in the context.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(question, retrieved_chunks):
    context = "\n\n---\n\n".join(retrieved_chunks)
    return PROMPT_TEMPLATE.format(context=context, question=question)

# quick test
sample_prompt = build_prompt("Why are people unhappy with Credit Cards?", test["documents"][:3])
print(sample_prompt[:500])

You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information. Do not invent details that are not present in the context.

Context:
credit card services harming the elderly and .

---

ments were due to a natural disaster in my inability to get to a bank. normally, i wouldve made a depo


In [11]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()
hf_token = os.environ.get("HF_TOKEN")
print("Token loaded:", bool(hf_token))

hf_client = InferenceClient(model="mistralai/Mistral-7B-Instruct-v0.2", token=hf_token)

def generate(prompt, max_new_tokens=300):
    try:
        response = hf_client.text_generation(prompt, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
        return response.strip()
    except Exception as e:
        return f"[Generator error: {e}]"

# quick test
print(generate(sample_prompt))

Token loaded: True
[Generator error: Model mistralai/Mistral-7B-Instruct-v0.2 is not supported for task text-generation and provider featherless-ai. Supported task: conversational.]


In [12]:
def answer(question, top_k=5, product_filter=None):
    retrieved = retrieve(question, top_k=top_k, product_filter=product_filter)
    prompt = build_prompt(question, retrieved["documents"])
    answer_text = generate(prompt)
    sources = [
        {"text": doc, "complaint_id": meta.get("complaint_id"),
         "product_category": meta.get("product_category"),
         "issue": meta.get("issue"), "company": meta.get("company")}
        for doc, meta in zip(retrieved["documents"], retrieved["metadatas"])
    ]
    return {"answer": answer_text, "sources": sources, "prompt": prompt}

# quick test
result = answer("Why are people unhappy with Credit Cards?")
print(result["answer"])

[Generator error: Model mistralai/Mistral-7B-Instruct-v0.2 is not supported for task text-generation and provider featherless-ai. Supported task: conversational.]


In [13]:
import pandas as pd

eval_questions = [
    "Why are people unhappy with Credit Cards?",
    "What are the most common complaints about money transfers?",
    "Are customers reporting unauthorized charges on their accounts?",
    "What issues do customers have with personal loans?",
    "Are there complaints about savings account fees?",
]

eval_records = []
for q in eval_questions:
    result = answer(q)
    top_sources = result["sources"][:2]
    source_summary = " | ".join(f"[{s['product_category']}] id={s['complaint_id']}: {s['text'][:100]}..." for s in top_sources)
    eval_records.append({
        "Question": q,
        "Generated Answer": result["answer"],
        "Retrieved Sources": source_summary,
        "Quality Score (1-5)": None,
        "Comments/Analysis": None,
    })

eval_df = pd.DataFrame(eval_records)
eval_df

,Question,Generated Answer,Retrieved Sources,Quality Score (1-5),Comments/Analysis
0,Why are people unhappy with Credit Cards?,[Generator error: Model mistralai/Mistral-7B-I...,[Credit Card] id=12682005: credit card service...,None,None
1,What are the most common complaints about mone...,[Generator error: Model mistralai/Mistral-7B-I...,[Money Transfer] id=11520794: problems with mo...,None,None
2,Are customers reporting unauthorized charges o...,[Generator error: Model mistralai/Mistral-7B-I...,[Credit Card] id=13166115: ection for customer...,None,None
3,What issues do customers have with personal lo...,[Generator error: Model mistralai/Mistral-7B-I...,[Personal Loan] id=12396109: ability and poor ...,None,None
4,Are there complaints about savings account fees?,[Generator error: Model mistralai/Mistral-7B-I...,[Savings Account] id=12967047: refund of all t...,None,None


In [14]:
eval_df.to_csv("../data/processed/task3_evaluation_raw.csv", index=False)
print("Saved.")

Saved.
